In [1]:
pip install transformers torch torchaudio datasets sentencepiece

In [2]:
import torch
import torchaudio
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)

In [3]:
# Load ASR Model -->wav2vec-XLS-R 300M
asr_model_id = "janiduchamika/wav2vec2-xls-r-300m-sinhala-general-185k"

asr_processor = Wav2Vec2Processor.from_pretrained(asr_model_id)
asr_model     = Wav2Vec2ForCTC.from_pretrained(asr_model_id)
asr_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/30.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec

In [19]:
# Load Correction Model -->T5
mt5_model_id = "Suchinthana/MT-5-Sinhala-Wikigen"
mt5_pipe = pipeline("text-generation", model=mt5_model_id)

mt5_tokenizer = AutoTokenizer.from_pretrained(mt5_model_id)
mt5_model     = AutoModelForSeq2SeqLM.from_pretrained(mt5_model_id)
mt5_model.eval()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'MT5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffL

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


MT5ForConditionalGeneration(
  (shared): Embedding(250112, 512)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 512)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
          

In [20]:
# Load and Resample Audio
def load_audio(path, target_sr=16000):
    waveform, sr = torchaudio.load(path)
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(sr, target_sr)
        waveform  = resampler(waveform)
    # wav2vec2 expects mono, 1D
    return waveform.squeeze().numpy()

In [21]:
# Helper: ASR model- Audio-->Raw text
def transcribe(audio_path):
    audio = load_audio(audio_path)
    inputs = asr_processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )
    with torch.no_grad():
        logits = asr_model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    return asr_processor.batch_decode(predicted_ids)[0]

In [22]:
# HELPER: T5 model-Raw text-->Corrected text
def correct_text(raw_text):
    prompt = f"writeWiki: {raw_text}"
    result = mt5_pipe(
        prompt,
        max_new_tokens=256,
        num_beams=4,
        early_stopping=True
    )
    return result[0]["generated_text"]

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
# Test Pipeline
audio_file = "/content/drive/MyDrive/Sentiment Analysis/test.wav"   # test audio

raw_transcript  = transcribe(audio_file)
clean_text      = correct_text(raw_transcript)

print("Raw ASR output  :", raw_transcript)
print("Corrected output:", clean_text)

Passing `generation_config` together with generation-related arguments=({'num_beams', 'early_stopping', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw ASR output  : අයිබවන් ම ප්‍රබත්මට පුළුවනි ොබර සහය වන්න ේංලින්දතකරබල ්බර්මදවස්පුනරේිවසෙක්් දසිතුන්නිමමලයල්දෙසැ්බර් විස් සිතුම්වෙනිද ඉඳලා නද විසිදරේහුකනක්සඑක හද දුන්නහැදන මවදසිබගේකදුවට පං ඔලිවව මටකක්ඔහටල්ක මේාධාර විදිහට අදෙවන්මට පුල්කමදාහරි ගැටළුව වෙලා තියෙන් ලෙස මේ ඇතුළත් කරපුවන්ක තුළමද නම කොහොමද  තේර්න මේකනක්ෂණිකමරයම විස්තර පරීක්ෂා කිරීමක් කරලා බැලුවස පහුගිය දෙසැ්බර් මාස විසි හත්වෙනිදා දිනේ තුළ ම ගැටුව පළිඳව පර කම්ප්ලේන එකක් දලා තියෙනවා  සාමාන්‍යයෙන් බිල්පත් අඩු කිරීමක ගැටළුව තිබුණු මාසේට අදාල බ්පතහරහාම වෙන්නේ නැැ එහෙ කරන්න හේතුව තියෙන්න ඒ විදිහටඅප බිල්පතරහා මුදලඅඩුකරොත්ගේ බදු මුදල අඩුවීම න්නේ  සාමා්‍ය ිල්පත් නිකුත් කරීමේ ක්‍රමය වේනේ දැන් ගැටළුව අවසානුනේස දැම්බර් මාසය තු දී  තකොට ඇයි ඊගාවර නිකුත් වෙන ිල්පත හරහා තමයි සැගේ දෙසැම්බර් මසේ වගේම නොවැම්බර් මාසේකණික්ෂ එක වැඩ නොකරපු කාල සීමා වේ බිල්පත් අඩු කිරීම වෙන්නඑතොට සකශනවාරි බිල්පතසගබලන් සයි කර් ෙකට අනුව තෙබරවාරි පළවිනිදා හෝ දෙවනිදා කියන ද දෙක තුළ නිකුත් වෙනා ඒබිල්පයන් තමයි ජස්මන් එක ලැබෙන් නෙස අ්මන්ට ක ලබාදීම සම්බන්ධව කිසිම ගැටළුවක්